# **2. Mapa Semântico dos Motivos de Cartão**

Este notebook implementa a análise semântica dos motivos de cartão usando a base limpa gerada pelo notebook 1.

Responsabilidade deste notebook:

- carregar `data/processed/card_reasons_long.csv`;
- transformar os motivos limpos em embeddings;
- clusterizar motivos semanticamente parecidos;
- projetar o mapa semântico em 2D;
- gerar tabelas de interpretação por cluster, ano, time e árbitro.

Todo parsing de PDF, extração de texto e limpeza estrutural das súmulas fica no `1-data_clean.ipynb`.

## **1. Configuração**

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

try:
    import umap
except ImportError:
    umap = None

try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    SentenceTransformer = None

pd.set_option("display.max_colwidth", 180)
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_PATH = PROJECT_ROOT / "data"
PROCESSED_PATH = DATA_PATH / "processed"
OUTPUT_DIR = PROCESSED_PATH / "semantic_card_reasons"

CARD_REASONS_LONG_PATH = PROCESSED_PATH / "card_reasons_long.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
TEXT_COLUMN = "reason_detail_key"  # texto limpo principal gerado no notebook 1

print(f"Project root: {PROJECT_ROOT}")
print(f"Base limpa existe? {CARD_REASONS_LONG_PATH.exists()}")

## **2. Carregamento da Base Limpa**

A entrada deste notebook é `card_reasons_long.csv`, criada no notebook 1. Cada linha representa um cartão e já contém o texto limpo do motivo.

A coluna principal para embedding é `reason_detail_key`, que reduz o peso de prefixos burocráticos como “Outro motivo...”. Quando a descrição livre não existe, o notebook 1 preenche essa coluna com a categoria limpa.

In [ ]:
REQUIRED_COLUMNS = {
    "year",
    "match_date",
    "home_team",
    "away_team",
    "arbitro",
    "card_type",
    "team",
    "reason",
    "reason_key",
    "reason_category",
    "reason_category_key",
    "reason_detail",
    "reason_detail_key",
}

if not CARD_REASONS_LONG_PATH.exists():
    raise FileNotFoundError(
        "Base limpa não encontrada. Rode primeiro o notebook 1-data_clean.ipynb "
        f"para gerar {CARD_REASONS_LONG_PATH}."
    )

reasons = pd.read_csv(CARD_REASONS_LONG_PATH)
missing = REQUIRED_COLUMNS.difference(reasons.columns)
if missing:
    raise ValueError(f"Colunas ausentes em card_reasons_long.csv: {sorted(missing)}")

reasons[TEXT_COLUMN] = reasons[TEXT_COLUMN].fillna("").astype(str).str.strip()
reasons = reasons[reasons[TEXT_COLUMN].ne("")].reset_index(drop=True)

print(f"Total de cartões com motivo: {len(reasons):,}")
print(f"Motivos completos únicos: {reasons['reason_key'].nunique():,}")
print(f"Textos para embedding únicos: {reasons[TEXT_COLUMN].nunique():,}")
print("\nTipos de cartão:")
print(reasons["card_type"].value_counts())

reasons.head()

## **3. Preparação dos Textos para Embedding**

Para não deixar frases repetidas dominarem o mapa, os embeddings são calculados sobre os textos únicos de `reason_detail_key`. Depois os clusters são propagados de volta para todos os cartões.

In [ ]:
reason_counts = reasons.groupby(TEXT_COLUMN).agg(
    reason=("reason", "first"),
    reason_category=("reason_category", "first"),
    reason_detail=("reason_detail", "first"),
    n_events=("reason", "size"),
    n_yellow=("card_type", lambda values: (values == "yellow").sum()),
    n_red=("card_type", lambda values: (values == "red").sum()),
).reset_index().rename(columns={TEXT_COLUMN: "embedding_key"})

reason_counts["text_for_embedding"] = reason_counts["embedding_key"]
reason_counts = reason_counts.sort_values("n_events", ascending=False).reset_index(drop=True)

print(f"Textos únicos para embedding: {len(reason_counts):,}")
reason_counts.head(15)

In [ ]:
PORTUGUESE_STOPWORDS = {
    "a", "ao", "aos", "as", "com", "como", "da", "das", "de", "do", "dos", "e", "em", "na", "nas",
    "no", "nos", "o", "os", "ou", "para", "por", "que", "se", "sem", "um", "uma", "uns", "umas",
    "adversario", "adversaria", "jogador", "maneira", "bola", "disputa", "dar", "tentar", "seu", "sua",
}


def build_tfidf_matrix(texts: pd.Series):
    vectorizer = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=1,
        max_df=0.95,
        stop_words=list(PORTUGUESE_STOPWORDS),
    )
    matrix = vectorizer.fit_transform(texts)
    return vectorizer, matrix

## **4. Embeddings Textuais**

O backend principal usa `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`, um encoder multilíngue adequado para frases curtas em português.

Se o modelo não puder ser carregado, o notebook usa um fallback com `TF-IDF + SVD`. Esse fallback também serve como baseline para discutir as limitações de bag-of-words apresentadas no PDF.

In [ ]:
texts = reason_counts["text_for_embedding"].tolist()
EMBEDDING_BACKEND = None
embedding_model = None
vectorizer = None
svd = None

if SentenceTransformer is not None:
    try:
        embedding_model = SentenceTransformer(MODEL_NAME)
        embeddings = embedding_model.encode(
            texts,
            show_progress_bar=True,
            normalize_embeddings=True,
        )
        embeddings = np.asarray(embeddings)
        EMBEDDING_BACKEND = "sentence-transformers"
    except Exception as error:
        print(f"Não foi possível carregar o SentenceTransformer: {error}")

if EMBEDDING_BACKEND is None:
    vectorizer, tfidf_matrix = build_tfidf_matrix(reason_counts["text_for_embedding"])
    max_components = min(50, tfidf_matrix.shape[0] - 1, tfidf_matrix.shape[1] - 1)
    n_components = max(2, max_components)
    svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)
    embeddings = normalize(svd.fit_transform(tfidf_matrix))
    EMBEDDING_BACKEND = "tfidf-svd"

print(f"Backend de embedding: {EMBEDDING_BACKEND}")
print(f"Matriz de embeddings: {embeddings.shape}")

## **5. Escolha do Número de Clusters**

A célula abaixo testa valores de `k` com silhouette score. O valor escolhido automaticamente é o melhor score dentro do intervalo, mas você pode sobrescrever `BEST_K` manualmente antes de rodar a clusterização.

In [ ]:
max_k = min(14, len(reason_counts) - 1)
candidate_ks = list(range(3, max_k + 1))
scores = []

for k in candidate_ks:
    labels = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=25).fit_predict(embeddings)
    score = silhouette_score(embeddings, labels, metric="cosine")
    scores.append({"k": k, "silhouette": score})

cluster_scores = pd.DataFrame(scores)
BEST_K = int(cluster_scores.sort_values("silhouette", ascending=False).iloc[0]["k"])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(cluster_scores["k"], cluster_scores["silhouette"], marker="o")
ax.axvline(BEST_K, color="crimson", linestyle="--", label=f"k escolhido = {BEST_K}")
ax.set_title("Silhouette por número de clusters")
ax.set_xlabel("Número de clusters")
ax.set_ylabel("Silhouette (cosine)")
ax.legend()
plt.show()

cluster_scores

In [ ]:
cluster_model = KMeans(n_clusters=BEST_K, random_state=RANDOM_STATE, n_init=50)
reason_counts["cluster"] = cluster_model.fit_predict(embeddings)

reasons_clustered = reasons.merge(
    reason_counts[["embedding_key", "cluster"]],
    left_on=TEXT_COLUMN,
    right_on="embedding_key",
    how="left",
).drop(columns=["embedding_key"])

print(reasons_clustered["cluster"].value_counts().sort_index())
reasons_clustered.head()

## **6. Resumo Interpretável dos Clusters**

Para nomear os grupos, usamos duas pistas:

1. exemplos mais frequentes de cada cluster;
2. termos e bigramas com maior TF-IDF dentro do cluster.

O nome final do cluster deve ser refinado manualmente depois de inspecionar a tabela.

In [ ]:
def top_terms_by_cluster(reason_table: pd.DataFrame, top_n: int = 8) -> pd.DataFrame:
    vectorizer_local, matrix = build_tfidf_matrix(reason_table["text_for_embedding"])
    feature_names = np.array(vectorizer_local.get_feature_names_out())
    rows = []
    for cluster_id, positions in reason_table.groupby("cluster").groups.items():
        cluster_matrix = matrix[list(positions)]
        weights = np.asarray(cluster_matrix.mean(axis=0)).ravel()
        top_positions = weights.argsort()[::-1][:top_n]
        rows.append({
            "cluster": cluster_id,
            "top_terms": ", ".join(feature_names[top_positions]),
        })
    return pd.DataFrame(rows)


def top_examples_by_cluster(reason_table: pd.DataFrame, top_n: int = 4) -> pd.DataFrame:
    rows = []
    for cluster_id, group in reason_table.groupby("cluster"):
        examples = (
            group.sort_values("n_events", ascending=False)
            .head(top_n)["reason"]
            .tolist()
        )
        rows.append({
            "cluster": cluster_id,
            "top_examples": " | ".join(examples),
        })
    return pd.DataFrame(rows)

cluster_terms = top_terms_by_cluster(reason_counts)
cluster_examples = top_examples_by_cluster(reason_counts)

cluster_summary = reason_counts.groupby("cluster").agg(
    n_unique_texts=("embedding_key", "size"),
    n_card_events=("n_events", "sum"),
    yellow_events=("n_yellow", "sum"),
    red_events=("n_red", "sum"),
).reset_index()
cluster_summary["red_pct"] = cluster_summary["red_events"] / cluster_summary["n_card_events"]
cluster_summary = (
    cluster_summary
    .merge(cluster_terms, on="cluster", how="left")
    .merge(cluster_examples, on="cluster", how="left")
    .sort_values("n_card_events", ascending=False)
    .reset_index(drop=True)
)

cluster_summary

## **7. Projeção 2D: Mapa Semântico**

A projeção 2D reduz os embeddings para duas dimensões. Pontos próximos representam motivos semanticamente parecidos. O tamanho do ponto indica quantas vezes aquele texto apareceu nas súmulas.

In [ ]:
if len(reason_counts) >= 5 and umap is not None:
    reducer = umap.UMAP(
        n_neighbors=min(15, len(reason_counts) - 1),
        min_dist=0.08,
        metric="cosine",
        random_state=RANDOM_STATE,
    )
    coords = reducer.fit_transform(embeddings)
    projection_method = "UMAP"
else:
    reducer = PCA(n_components=2, random_state=RANDOM_STATE)
    coords = reducer.fit_transform(embeddings)
    projection_method = "PCA"

reason_counts["x"] = coords[:, 0]
reason_counts["y"] = coords[:, 1]

fig, ax = plt.subplots(figsize=(11, 8))
for cluster_id, group in reason_counts.groupby("cluster"):
    ax.scatter(
        group["x"],
        group["y"],
        s=np.clip(group["n_events"] * 6, 25, 350),
        alpha=0.72,
        label=f"Cluster {cluster_id}",
    )

for cluster_id, group in reason_counts.groupby("cluster"):
    centroid = group[["x", "y"]].mean()
    ax.text(
        centroid["x"],
        centroid["y"],
        str(cluster_id),
        fontsize=12,
        fontweight="bold",
        ha="center",
        va="center",
        bbox={"boxstyle": "round,pad=0.25", "facecolor": "white", "alpha": 0.75, "edgecolor": "0.7"},
    )

ax.set_title(f"Mapa semântico dos motivos de cartão ({projection_method})")
ax.set_xlabel("Dimensão 1")
ax.set_ylabel("Dimensão 2")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Clusters")
plt.tight_layout()
plt.show()

In [ ]:
fig_path = OUTPUT_DIR / "card_reason_semantic_map.png"

fig, ax = plt.subplots(figsize=(11, 8))
for cluster_id, group in reason_counts.groupby("cluster"):
    ax.scatter(
        group["x"],
        group["y"],
        s=np.clip(group["n_events"] * 6, 25, 350),
        alpha=0.72,
        label=f"Cluster {cluster_id}",
    )
for cluster_id, group in reason_counts.groupby("cluster"):
    centroid = group[["x", "y"]].mean()
    ax.text(
        centroid["x"],
        centroid["y"],
        str(cluster_id),
        fontsize=12,
        fontweight="bold",
        ha="center",
        va="center",
        bbox={"boxstyle": "round,pad=0.25", "facecolor": "white", "alpha": 0.75, "edgecolor": "0.7"},
    )
ax.set_title(f"Mapa semântico dos motivos de cartão ({projection_method})")
ax.set_xlabel("Dimensão 1")
ax.set_ylabel("Dimensão 2")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Clusters")
plt.tight_layout()
fig.savefig(fig_path, dpi=180, bbox_inches="tight")
plt.close(fig)

print(f"Figura salva em: {fig_path}")

## **8. Leituras por Ano, Time e Árbitro**

Depois de rotular cada cartão com um cluster semântico, dá para comparar a distribuição dos tipos de punição por ano, time e árbitro.

In [ ]:
cluster_by_year = pd.crosstab(
    reasons_clustered["year"],
    reasons_clustered["cluster"],
    normalize="index",
).round(3)

cluster_by_year

In [ ]:
top_teams = reasons_clustered["team"].value_counts().head(12).index
cluster_by_team = pd.crosstab(
    reasons_clustered[reasons_clustered["team"].isin(top_teams)]["team"],
    reasons_clustered[reasons_clustered["team"].isin(top_teams)]["cluster"],
    normalize="index",
).round(3)

cluster_by_team

In [ ]:
top_referees = reasons_clustered["arbitro"].value_counts().head(12).index
cluster_by_referee = pd.crosstab(
    reasons_clustered[reasons_clustered["arbitro"].isin(top_referees)]["arbitro"],
    reasons_clustered[reasons_clustered["arbitro"].isin(top_referees)]["cluster"],
    normalize="index",
).round(3)

cluster_by_referee

## **9. Busca Semântica de Motivos Parecidos**

Esta função consulta o espaço vetorial. Exemplos úteis:

- `semantic_search("reclamação contra arbitragem")`
- `semantic_search("entrada temerária na disputa de bola")`
- `semantic_search("retardar o reinício do jogo")`
- `semantic_search("conduta violenta")`

In [ ]:
def embed_query(texts_to_embed: list[str]) -> np.ndarray:
    if EMBEDDING_BACKEND == "sentence-transformers":
        return np.asarray(embedding_model.encode(texts_to_embed, normalize_embeddings=True))
    query_matrix = vectorizer.transform(texts_to_embed)
    return normalize(svd.transform(query_matrix))


def semantic_search(query: str, top_k: int = 10) -> pd.DataFrame:
    query_embedding = embed_query([query])
    similarities = cosine_similarity(query_embedding, embeddings).ravel()
    result = reason_counts.copy()
    result["similarity"] = similarities
    return result.sort_values("similarity", ascending=False)[[
        "cluster",
        "similarity",
        "n_events",
        "n_yellow",
        "n_red",
        "reason_category",
        "reason_detail",
        "reason",
    ]].head(top_k)

semantic_search("reclamação contra arbitragem", top_k=10)

## **10. Salvamento dos Artefatos**

In [ ]:
reason_counts_path = OUTPUT_DIR / "card_reason_unique_clusters.csv"
reasons_clustered_path = OUTPUT_DIR / "card_reasons_long_with_clusters.csv"
cluster_summary_path = OUTPUT_DIR / "card_reason_cluster_summary.csv"
cluster_by_year_path = OUTPUT_DIR / "card_reason_clusters_by_year.csv"
cluster_by_team_path = OUTPUT_DIR / "card_reason_clusters_by_team_top12.csv"
cluster_by_referee_path = OUTPUT_DIR / "card_reason_clusters_by_referee_top12.csv"

reason_counts.to_csv(reason_counts_path, index=False)
reasons_clustered.to_csv(reasons_clustered_path, index=False)
cluster_summary.to_csv(cluster_summary_path, index=False)
cluster_by_year.to_csv(cluster_by_year_path)
cluster_by_team.to_csv(cluster_by_team_path)
cluster_by_referee.to_csv(cluster_by_referee_path)

print("Arquivos salvos:")
for path in [
    reason_counts_path,
    reasons_clustered_path,
    cluster_summary_path,
    cluster_by_year_path,
    cluster_by_team_path,
    cluster_by_referee_path,
    fig_path,
]:
    print(f"- {path}")

## **11. Próximos Passos Analíticos**

Depois de rodar o notebook, revise `cluster_summary` e dê nomes substantivos aos clusters, por exemplo:

- falta temerária;
- reclamação/protesto;
- retardar reinício;
- impedir ataque promissor;
- conduta violenta;
- segundo amarelo/expulsão.

Esses nomes podem virar uma coluna manual `cluster_label`, útil para gráficos finais e para transformar o mapa semântico em uma análise disciplinar interpretável.